In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_l3869_suspension_interest_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema=(
        #    "L3869SuspensionInterestId string,"
        #    "SourceSystem string,"
        #    "PortfolioKey string,"
        #    "PortfolioName string,"
        #    "IngestionTimestamp timestamp,"
        #    "BatchId string,"
        #    "L3869SuspensionInterestKey string,"
        #    "ExposureKey string,"
        #    "PartyKey string,"
        #    "CDI string,"
        #    "Interest string,"
        #    "PenaltyInterest string,"
        #    "StartDate string,"
        #    "BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.L3869SuspensionInterest"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== profits_user_defined_fields ==================

        user_defined_fields_extra_filters = None

        if user_defined_fields_extra_filters is not None:
            user_defined_fields_extra_filters.append(date_filter)
            user_defined_fields_final_filters = " AND ".join(
                user_defined_fields_extra_filters
            )
        else:
            user_defined_fields_final_filters = date_filter

        user_defined_fields_selected_fields = [
            "UD_ACCOUNT_NUMBER",
            "FIELD_VALUE",
            "PFG_TAG",
            "PFG_TAG_SET_CODE",
            "BusinessDate",
        ]

        user_defined_fields = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_user_defined_fields",
            user_defined_fields_final_filters,
            user_defined_fields_selected_fields,
        )

        # ================== profits_loan_details ==================

        loan_details_extra_filters = None

        if loan_details_extra_filters is not None:
            loan_details_extra_filters.append(date_filter)
            loan_details_final_filters = " AND ".join(loan_details_extra_filters)
        else:
            loan_details_final_filters = date_filter  # possible need change date

        loan_details_selected_fields = [
            "ACCOUNT_NUMBER",
            "agreement_number",
            "prft_acc_unit",
        ]

        loan_details = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_details",
            loan_details_final_filters,
            loan_details_selected_fields,
        )
        loan_details = loan_details.withColumnRenamed(
            "ACCOUNT_NUMBER", "account_number_ld"
        )
        # ================== profits_agreement ==================

        # agreement_extra_filters = None

        # if agreement_extra_filters is not None:
        #    agreement_extra_filters.append(date_filter)
        #    agreement_final_filters = " AND ".join(agreement_extra_filters)
        # else:
        #   agreement_final_filters = date_filter

        #   agreement_selected_fields = [
        #       "agreement_number",
        #       "Unit_code"
        #   ]

        #   agreement = load_landed_delta_table(
        #       spark,
        #       catalog_landed,  # type: ignore
        #       "profits_agreement",
        #       agreement_final_filters,
        #       agreement_selected_fields,
        #   )

        # ================== profits_customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

            customer_selected_fields = ["CUST_ID", "CDI"]

            customer = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "profits_customer",
                customer_final_filters,
                customer_selected_fields,
            )

        # ==================== portfolio =====================

        portfolio_spread_extra_filters = None

        if portfolio_spread_extra_filters is not None:
            portfolio_spread_extra_filters.append(date_filter)
            portfolio_spread_final_filters = " AND ".join(
                portfolio_spread_extra_filters
            )
        else:
            portfolio_spread_final_filters = date_filter

        portfolio_spread_selected_fields = ["PortfolioName", "unit_code_profits"]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_spread_final_filters,
            portfolio_spread_selected_fields,
        )

        # ====================  suspension_1_df =========================

        # Rename multiple column fields to avoid issues that causes ambiguity

        user_defined_fields_suspension_1 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_prime")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_prime")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_prime")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_prime")
        )

        user_defined_fields_t1_suspension_1 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_t1")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_t1")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_t1")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_t1")
        )

        user_defined_fields_t2_suspension_1 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_t2")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_t2")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_t2")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_t2")
        )

        user_defined_fields_dt_suspension_1 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_dt")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_dt")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_dt")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_dt")
        )

        # Join user_defined_fields with itself

        user_defined_fields_suspension_1 = user_defined_fields_suspension_1.filter(
            (user_defined_fields_suspension_1.PFG_TAG_prime == "PFTAG92")
            & (
                user_defined_fields_suspension_1.PFG_TAG_SET_CODE_prime
                == "ΤΟΚΟΙ Ν.3869"
            )
        )

        user_defined_fields_dt_suspension_1_join_condition = (
            (
                user_defined_fields_suspension_1.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_dt_suspension_1.UD_ACCOUNT_NUMBER_dt
            )
            & (
                user_defined_fields_dt_suspension_1.PFG_TAG_SET_CODE_dt
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_dt_suspension_1.PFG_TAG_dt == "PFTAG93")
        )

        user_defined_fields_t1_suspension_1_join_condition = (
            (
                user_defined_fields_suspension_1.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_t1_suspension_1.UD_ACCOUNT_NUMBER_t1
            )
            & (
                user_defined_fields_t1_suspension_1.PFG_TAG_SET_CODE_t1
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_t1_suspension_1.PFG_TAG_t1 == "PFTAG94")
        )

        user_defined_fields_t2_suspension_1_join_condition = (
            (
                user_defined_fields_suspension_1.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_t2_suspension_1.UD_ACCOUNT_NUMBER_t2
            )
            & (
                user_defined_fields_t2_suspension_1.PFG_TAG_SET_CODE_t2
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_t2_suspension_1.PFG_TAG_t2 == "PFTAG95")
        )

        portfolio_loan_details_join_condition = (
            portfolio.unit_code_profits == loan_details.prft_acc_unit
        )

        suspension_1_df = (
            user_defined_fields_suspension_1.join(
                user_defined_fields_dt_suspension_1,
                user_defined_fields_dt_suspension_1_join_condition,
                "inner",
            )
            .join(
                user_defined_fields_t1_suspension_1,
                user_defined_fields_t1_suspension_1_join_condition,
                "inner",
            )
            .join(
                user_defined_fields_t2_suspension_1,
                user_defined_fields_t2_suspension_1_join_condition,
                "inner",
            )
        )

        suspension_1_df = suspension_1_df.select(
            user_defined_fields_suspension_1.UD_ACCOUNT_NUMBER_prime.alias(
                "ACCOUNT_NUMBER"
            ),
            user_defined_fields_suspension_1.FIELD_VALUE_prime.alias("cust_id_1"),
            user_defined_fields_dt_suspension_1.FIELD_VALUE_dt.alias("StartDate"),
            user_defined_fields_t1_suspension_1.FIELD_VALUE_t1.alias("InterestAmount"),
            user_defined_fields_t2_suspension_1.FIELD_VALUE_t2.alias("PenaltyAmount"),
            user_defined_fields_suspension_1.PFG_TAG_prime.alias("PFG_TAG"),
            user_defined_fields_suspension_1.BusinessDate.alias("businessdate"),
        )

        suspension_1_df = suspension_1_df.join(
            customer, suspension_1_df.cust_id_1 == customer.CUST_ID, "inner"
        )

        # creating another table from user_defined_fields with some specific filter conditions in order to satisfy the where exists criterion

        user_defined_fields_a_1 = user_defined_fields.filter(
            (user_defined_fields.PFG_TAG == "PFTAG141")
            & (user_defined_fields.FIELD_VALUE.isin(["04", "05", "20"]))
        )

        user_defined_fields_a_1 = user_defined_fields_a_1.withColumn(
            "UD_ACCOUNT_NUMBER_INT",
            user_defined_fields_a_1.UD_ACCOUNT_NUMBER.cast("int"),
        )

        # join conditions
        suspension_1_df_loan_details_join_condition = (
            suspension_1_df.ACCOUNT_NUMBER == loan_details.account_number_ld
        )
        suspension_df_1_df_user_defined_fields_a_1_join_condition = (
            user_defined_fields_a_1.UD_ACCOUNT_NUMBER_INT == suspension_1_df.cust_id_1
        )

        # loan_details_agreement_join_condition=(loan_details.agreement_number == agreement.agreement_number)

        suspension_1_df = (
            suspension_1_df.join(
                user_defined_fields_a_1,
                suspension_df_1_df_user_defined_fields_a_1_join_condition,
                "left_semi",
            )
            .join(loan_details, suspension_1_df_loan_details_join_condition, "inner")
            .join(
                portfolio,
                portfolio_loan_details_join_condition,
                "left",
            )
        )
        # .join(
        #    agreement,
        #    loan_details_agreement_join_condition,
        #    "left"
        # )

        L3869SuspensionInterestId_1_primary_key_fields = [
            suspension_1_df.ACCOUNT_NUMBER,
            suspension_1_df.CUST_ID,
            lit("profits"),
        ]

        L3869SuspensionInterestKey_1_primary_key_fields = [
            suspension_1_df.ACCOUNT_NUMBER,
            suspension_1_df.CUST_ID,
        ]

        suspension_1_df = suspension_1_df.withColumn(
            "L3869SuspensionInterestId",
            sha2(concat_ws(";", *L3869SuspensionInterestId_1_primary_key_fields), 256),
        ).withColumn(
            "L3869SuspensionInterestKey",
            concat_ws(";", *L3869SuspensionInterestKey_1_primary_key_fields),
        )

        suspension_1_target_df = suspension_1_df.select(
            suspension_1_df.L3869SuspensionInterestId.alias(
                "L3869SuspensionInterestId"
            ),
            lit("profits").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            suspension_1_df.L3869SuspensionInterestKey.alias(
                "L3869SuspensionInterestKey"
            ),
            suspension_1_df.ACCOUNT_NUMBER.cast("string").alias("ExposureKey"),
            suspension_1_df.CUST_ID.cast("string").alias("PartyKey"),
            suspension_1_df.CDI.alias("CDI"),
            suspension_1_df.InterestAmount.cast("string").alias("Interest"),
            suspension_1_df.PenaltyAmount.cast("string").alias("PenaltyInterest"),
            suspension_1_df.StartDate.alias("StartDate"),
            suspension_1_df.businessdate.alias("BusinessDate"),
        )

        # =======================  suspension_2_df =============================

        # Rename multiple column fields to avoid issues that causes ambiguity

        user_defined_fields_suspension_2 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_prime")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_prime")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_prime")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_prime")
        )

        user_defined_fields_t1_suspension_2 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_t1")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_t1")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_t1")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_t1")
        )

        user_defined_fields_t2_suspension_2 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_t2")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_t2")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_t2")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_t2")
        )

        user_defined_fields_dt_suspension_2 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_dt")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_dt")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_dt")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_dt")
        )

        # Join user_defined_fields with itself

        user_defined_fields_suspension_2 = user_defined_fields_suspension_2.filter(
            (user_defined_fields_suspension_2.PFG_TAG_prime == "PFTAG96")
            & (
                user_defined_fields_suspension_2.PFG_TAG_SET_CODE_prime
                == "ΤΟΚΟΙ Ν.3869"
            )
        )

        user_defined_fields_dt_suspension_2_join_condition = (
            (
                user_defined_fields_suspension_2.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_dt_suspension_2.UD_ACCOUNT_NUMBER_dt
            )
            & (
                user_defined_fields_dt_suspension_2.PFG_TAG_SET_CODE_dt
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_dt_suspension_2.PFG_TAG_dt == "PFTAG97")
        )

        user_defined_fields_t1_suspension_2_join_condition = (
            (
                user_defined_fields_suspension_2.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_t1_suspension_2.UD_ACCOUNT_NUMBER_t1
            )
            & (
                user_defined_fields_t1_suspension_2.PFG_TAG_SET_CODE_t1
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_t1_suspension_2.PFG_TAG_t1 == "PFTAG98")
        )

        user_defined_fields_t2_suspension_2_join_condition = (
            (
                user_defined_fields_suspension_2.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_t2_suspension_2.UD_ACCOUNT_NUMBER_t2
            )
            & (
                user_defined_fields_t2_suspension_2.PFG_TAG_SET_CODE_t2
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_t2_suspension_2.PFG_TAG_t2 == "PFTAG99")
        )

        portfolio_loan_details_join_condition = (
            portfolio.unit_code_profits == loan_details.prft_acc_unit
        )

        suspension_2_df = (
            user_defined_fields_suspension_2.join(
                user_defined_fields_dt_suspension_2,
                user_defined_fields_dt_suspension_2_join_condition,
                "inner",
            )
            .join(
                user_defined_fields_t1_suspension_2,
                user_defined_fields_t1_suspension_2_join_condition,
                "inner",
            )
            .join(
                user_defined_fields_t2_suspension_2,
                user_defined_fields_t2_suspension_2_join_condition,
                "inner",
            )
        )

        suspension_2_df = suspension_2_df.select(
            user_defined_fields_suspension_2.UD_ACCOUNT_NUMBER_prime.alias(
                "ACCOUNT_NUMBER"
            ),
            user_defined_fields_suspension_2.FIELD_VALUE_prime.alias("cust_id_2"),
            user_defined_fields_dt_suspension_2.FIELD_VALUE_dt.alias("StartDate"),
            user_defined_fields_t1_suspension_2.FIELD_VALUE_t1.alias("InterestAmount"),
            user_defined_fields_t2_suspension_2.FIELD_VALUE_t2.alias("PenaltyAmount"),
            user_defined_fields_suspension_2.PFG_TAG_prime.alias("PFG_TAG"),
            user_defined_fields_suspension_2.BusinessDate.alias("businessdate"),
        )

        suspension_2_df = suspension_2_df.join(
            customer, suspension_2_df.cust_id_2 == customer.CUST_ID, "inner"
        )

        # creating another table from user_defined_fields with some specific filter conditions in order to satisfy the where exists criterion

        user_defined_fields_a_2 = user_defined_fields.filter(
            (user_defined_fields.PFG_TAG == "PFTAG141")
            & (user_defined_fields.FIELD_VALUE.isin(["04", "05", "20"]))
        )

        user_defined_fields_a_2 = user_defined_fields_a_2.withColumn(
            "UD_ACCOUNT_NUMBER_INT",
            user_defined_fields_a_2.UD_ACCOUNT_NUMBER.cast("int"),
        )

        # join conditions
        suspension_2_df_loan_details_join_condition = (
            suspension_2_df.ACCOUNT_NUMBER == loan_details.account_number_ld
        )
        suspension_df_2_df_user_defined_fields_a_2_join_condition = (
            user_defined_fields_a_2.UD_ACCOUNT_NUMBER_INT == suspension_2_df.cust_id_2
        )

        # loan_details_agreement_join_condition=(loan_details.agreement_number == agreement.agreement_number)

        suspension_2_df = (
            suspension_2_df.join(
                user_defined_fields_a_2,
                suspension_df_2_df_user_defined_fields_a_2_join_condition,
                "left_semi",
            )
            .join(loan_details, suspension_2_df_loan_details_join_condition, "inner")
            .join(
                portfolio,
                portfolio_loan_details_join_condition,
                "left",
            )
        )
        # .join(
        #    agreement,
        #    loan_details_agreement_join_condition,
        #    "left"
        # )

        L3869SuspensionInterestId_2_primary_key_fields = [
            suspension_2_df.ACCOUNT_NUMBER,
            suspension_2_df.CUST_ID,
            lit("profits"),
        ]

        L3869SuspensionInterestKey_2_primary_key_fields = [
            suspension_2_df.ACCOUNT_NUMBER,
            suspension_2_df.CUST_ID,
        ]

        suspension_2_df = suspension_2_df.withColumn(
            "L3869SuspensionInterestId",
            sha2(concat_ws(";", *L3869SuspensionInterestId_2_primary_key_fields), 256),
        ).withColumn(
            "L3869SuspensionInterestKey",
            concat_ws(";", *L3869SuspensionInterestKey_2_primary_key_fields),
        )

        suspension_2_target_df = suspension_2_df.select(
            suspension_2_df.L3869SuspensionInterestId.alias(
                "L3869SuspensionInterestId"
            ),
            lit("profits").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            suspension_2_df.L3869SuspensionInterestKey.alias(
                "L3869SuspensionInterestKey"
            ),
            suspension_2_df.ACCOUNT_NUMBER.cast("string").alias("ExposureKey"),
            suspension_2_df.CUST_ID.cast("string").alias("PartyKey"),
            suspension_2_df.CDI.alias("CDI"),
            suspension_2_df.InterestAmount.cast("string").alias("Interest"),
            suspension_2_df.PenaltyAmount.cast("string").alias("PenaltyInterest"),
            suspension_2_df.StartDate.alias("StartDate"),
            suspension_2_df.businessdate.alias("BusinessDate"),
        )

        # =================================  suspension_3_df ====================================

        # Rename multiple column fields to avoid issues that causes ambiguity

        user_defined_fields_suspension_3 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_prime")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_prime")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_prime")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_prime")
        )

        user_defined_fields_t1_suspension_3 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_t1")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_t1")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_t1")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_t1")
        )

        user_defined_fields_t2_suspension_3 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_t2")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_t2")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_t2")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_t2")
        )

        user_defined_fields_dt_suspension_3 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_dt")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_dt")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_dt")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_dt")
        )

        # Join user_defined_fields with itself

        user_defined_fields_suspension_3 = user_defined_fields_suspension_3.filter(
            (user_defined_fields_suspension_3.PFG_TAG_prime == "PFTAG101")
            & (
                user_defined_fields_suspension_3.PFG_TAG_SET_CODE_prime
                == "ΤΟΚΟΙ Ν.3869"
            )
        )

        user_defined_fields_dt_suspension_3_join_condition = (
            (
                user_defined_fields_suspension_3.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_dt_suspension_3.UD_ACCOUNT_NUMBER_dt
            )
            & (
                user_defined_fields_dt_suspension_3.PFG_TAG_SET_CODE_dt
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_dt_suspension_3.PFG_TAG_dt == "PFTAG102")
        )

        user_defined_fields_t1_suspension_3_join_condition = (
            (
                user_defined_fields_suspension_3.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_t1_suspension_3.UD_ACCOUNT_NUMBER_t1
            )
            & (
                user_defined_fields_t1_suspension_3.PFG_TAG_SET_CODE_t1
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_t1_suspension_3.PFG_TAG_t1 == "PFTAG103")
        )

        user_defined_fields_t2_suspension_3_join_condition = (
            (
                user_defined_fields_suspension_3.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_t2_suspension_3.UD_ACCOUNT_NUMBER_t2
            )
            & (
                user_defined_fields_t2_suspension_3.PFG_TAG_SET_CODE_t2
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_t2_suspension_3.PFG_TAG_t2 == "PFTAG104")
        )

        portfolio_loan_details_join_condition = (
            portfolio.unit_code_profits == loan_details.prft_acc_unit
        )

        suspension_3_df = (
            user_defined_fields_suspension_3.join(
                user_defined_fields_dt_suspension_3,
                user_defined_fields_dt_suspension_3_join_condition,
                "inner",
            )
            .join(
                user_defined_fields_t1_suspension_3,
                user_defined_fields_t1_suspension_3_join_condition,
                "inner",
            )
            .join(
                user_defined_fields_t2_suspension_3,
                user_defined_fields_t2_suspension_3_join_condition,
                "inner",
            )
        )

        suspension_3_df = suspension_3_df.select(
            user_defined_fields_suspension_3.UD_ACCOUNT_NUMBER_prime.alias(
                "ACCOUNT_NUMBER"
            ),
            user_defined_fields_suspension_3.FIELD_VALUE_prime.alias("cust_id_3"),
            user_defined_fields_dt_suspension_3.FIELD_VALUE_dt.alias("StartDate"),
            user_defined_fields_t1_suspension_3.FIELD_VALUE_t1.alias("InterestAmount"),
            user_defined_fields_t2_suspension_3.FIELD_VALUE_t2.alias("PenaltyAmount"),
            user_defined_fields_suspension_3.PFG_TAG_prime.alias("PFG_TAG"),
            user_defined_fields_suspension_3.BusinessDate.alias("businessdate"),
        )

        suspension_3_df = suspension_3_df.join(
            customer, suspension_3_df.cust_id_3 == customer.CUST_ID, "inner"
        )

        # creating another table from user_defined_fields with some specific filter conditions in order to satisfy the where exists criterion

        user_defined_fields_a_3 = user_defined_fields.filter(
            (user_defined_fields.PFG_TAG == "PFTAG141")
            & (user_defined_fields.FIELD_VALUE.isin(["04", "05", "20"]))
        )

        user_defined_fields_a_3 = user_defined_fields_a_3.withColumn(
            "UD_ACCOUNT_NUMBER_INT",
            user_defined_fields_a_3.UD_ACCOUNT_NUMBER.cast("int"),
        )

        # join conditions
        suspension_3_df_loan_details_join_condition = (
            suspension_3_df.ACCOUNT_NUMBER == loan_details.account_number_ld
        )
        suspension_df_3_df_user_defined_fields_a_3_join_condition = (
            user_defined_fields_a_3.UD_ACCOUNT_NUMBER_INT == suspension_3_df.cust_id_3
        )

        # loan_details_agreement_join_condition=(loan_details.agreement_number == agreement.agreement_number)

        suspension_3_df = (
            suspension_3_df.join(
                user_defined_fields_a_3,
                suspension_df_3_df_user_defined_fields_a_3_join_condition,
                "left_semi",
            )
            .join(loan_details, suspension_3_df_loan_details_join_condition, "inner")
            .join(
                portfolio,
                portfolio_loan_details_join_condition,
                "left",
            )
        )
        # .join(
        #    agreement,
        #    loan_details_agreement_join_condition,
        #    "left"
        # )

        L3869SuspensionInterestId_3_primary_key_fields = [
            suspension_3_df.ACCOUNT_NUMBER,
            suspension_3_df.CUST_ID,
            lit("profits"),
        ]

        L3869SuspensionInterestKey_3_primary_key_fields = [
            suspension_3_df.ACCOUNT_NUMBER,
            suspension_3_df.CUST_ID,
        ]

        suspension_3_df = suspension_3_df.withColumn(
            "L3869SuspensionInterestId",
            sha2(concat_ws(";", *L3869SuspensionInterestId_3_primary_key_fields), 256),
        ).withColumn(
            "L3869SuspensionInterestKey",
            concat_ws(";", *L3869SuspensionInterestKey_3_primary_key_fields),
        )

        suspension_3_target_df = suspension_3_df.select(
            suspension_3_df.L3869SuspensionInterestId.alias(
                "L3869SuspensionInterestId"
            ),
            lit("profits").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            suspension_3_df.L3869SuspensionInterestKey.alias(
                "L3869SuspensionInterestKey"
            ),
            suspension_3_df.ACCOUNT_NUMBER.cast("string").alias("ExposureKey"),
            suspension_3_df.CUST_ID.cast("string").alias("PartyKey"),
            suspension_3_df.CDI.alias("CDI"),
            suspension_3_df.InterestAmount.cast("string").alias("Interest"),
            suspension_3_df.PenaltyAmount.cast("string").alias("PenaltyInterest"),
            suspension_3_df.StartDate.alias("StartDate"),
            suspension_3_df.businessdate.alias("BusinessDate"),
        )

        # ===============================  suspension_4_df =====================================

        # Rename multiple column fields to avoid issues that causes ambiguity

        user_defined_fields_suspension_4 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_prime")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_prime")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_prime")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_prime")
        )

        user_defined_fields_t1_suspension_4 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_t1")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_t1")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_t1")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_t1")
        )

        user_defined_fields_t2_suspension_4 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_t2")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_t2")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_t2")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_t2")
        )

        user_defined_fields_dt_suspension_4 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_dt")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_dt")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_dt")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_dt")
        )

        # Join user_defined_fields with itself

        user_defined_fields_suspension_4 = user_defined_fields_suspension_4.filter(
            (user_defined_fields_suspension_4.PFG_TAG_prime == "PFTAG105")
            & (
                user_defined_fields_suspension_4.PFG_TAG_SET_CODE_prime
                == "ΤΟΚΟΙ Ν.3869"
            )
        )

        user_defined_fields_dt_suspension_4_join_condition = (
            (
                user_defined_fields_suspension_4.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_dt_suspension_4.UD_ACCOUNT_NUMBER_dt
            )
            & (
                user_defined_fields_dt_suspension_4.PFG_TAG_SET_CODE_dt
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_dt_suspension_4.PFG_TAG_dt == "PFTAG106")
        )

        user_defined_fields_t1_suspension_4_join_condition = (
            (
                user_defined_fields_suspension_4.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_t1_suspension_4.UD_ACCOUNT_NUMBER_t1
            )
            & (
                user_defined_fields_t1_suspension_4.PFG_TAG_SET_CODE_t1
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_t1_suspension_4.PFG_TAG_t1 == "PFTAG107")
        )

        user_defined_fields_t2_suspension_4_join_condition = (
            (
                user_defined_fields_suspension_4.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_t2_suspension_4.UD_ACCOUNT_NUMBER_t2
            )
            & (
                user_defined_fields_t2_suspension_4.PFG_TAG_SET_CODE_t2
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_t2_suspension_4.PFG_TAG_t2 == "PFTAG108")
        )

        portfolio_loan_details_join_condition = (
            portfolio.unit_code_profits == loan_details.prft_acc_unit
        )

        suspension_4_df = (
            user_defined_fields_suspension_4.join(
                user_defined_fields_dt_suspension_4,
                user_defined_fields_dt_suspension_4_join_condition,
                "inner",
            )
            .join(
                user_defined_fields_t1_suspension_4,
                user_defined_fields_t1_suspension_4_join_condition,
                "inner",
            )
            .join(
                user_defined_fields_t2_suspension_4,
                user_defined_fields_t2_suspension_4_join_condition,
                "inner",
            )
        )

        suspension_4_df = suspension_4_df.select(
            user_defined_fields_suspension_4.UD_ACCOUNT_NUMBER_prime.alias(
                "ACCOUNT_NUMBER"
            ),
            user_defined_fields_suspension_4.FIELD_VALUE_prime.alias("cust_id_4"),
            user_defined_fields_dt_suspension_4.FIELD_VALUE_dt.alias("StartDate"),
            user_defined_fields_t1_suspension_4.FIELD_VALUE_t1.alias("InterestAmount"),
            user_defined_fields_t2_suspension_4.FIELD_VALUE_t2.alias("PenaltyAmount"),
            user_defined_fields_suspension_4.PFG_TAG_prime.alias("PFG_TAG"),
            user_defined_fields_suspension_4.BusinessDate.alias("businessdate"),
        )

        suspension_4_df = suspension_4_df.join(
            customer, suspension_4_df.cust_id_4 == customer.CUST_ID, "inner"
        )

        # creating another table from user_defined_fields with some specific filter conditions in order to satisfy the where exists criterion

        user_defined_fields_a_4 = user_defined_fields.filter(
            (user_defined_fields.PFG_TAG == "PFTAG141")
            & (user_defined_fields.FIELD_VALUE.isin(["04", "05", "20"]))
        )

        user_defined_fields_a_4 = user_defined_fields_a_4.withColumn(
            "UD_ACCOUNT_NUMBER_INT",
            user_defined_fields_a_4.UD_ACCOUNT_NUMBER.cast("int"),
        )

        # join conditions
        suspension_4_df_loan_details_join_condition = (
            suspension_4_df.ACCOUNT_NUMBER == loan_details.account_number_ld
        )
        suspension_df_4_df_user_defined_fields_a_4_join_condition = (
            user_defined_fields_a_4.UD_ACCOUNT_NUMBER_INT == suspension_4_df.cust_id_4
        )

        # loan_details_agreement_join_condition=(loan_details.agreement_number == agreement.agreement_number)

        suspension_4_df = (
            suspension_4_df.join(
                user_defined_fields_a_4,
                suspension_df_4_df_user_defined_fields_a_4_join_condition,
                "left_semi",
            )
            .join(loan_details, suspension_4_df_loan_details_join_condition, "inner")
            .join(
                portfolio,
                portfolio_loan_details_join_condition,
                "left",
            )
        )
        # .join(
        #    agreement,
        #    loan_details_agreement_join_condition,
        #    "left"
        # )

        L3869SuspensionInterestId_4_primary_key_fields = [
            suspension_4_df.ACCOUNT_NUMBER,
            suspension_4_df.CUST_ID,
            lit("profits"),
        ]

        L3869SuspensionInterestKey_4_primary_key_fields = [
            suspension_4_df.ACCOUNT_NUMBER,
            suspension_4_df.CUST_ID,
        ]

        suspension_4_df = suspension_4_df.withColumn(
            "L3869SuspensionInterestId",
            sha2(concat_ws(";", *L3869SuspensionInterestId_4_primary_key_fields), 256),
        ).withColumn(
            "L3869SuspensionInterestKey",
            concat_ws(";", *L3869SuspensionInterestKey_4_primary_key_fields),
        )

        suspension_4_target_df = suspension_4_df.select(
            suspension_4_df.L3869SuspensionInterestId.alias(
                "L3869SuspensionInterestId"
            ),
            lit("profits").alias("SourceSystem"),
            portfolio.unit_code_profits.alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            suspension_4_df.L3869SuspensionInterestKey.alias(
                "L3869SuspensionInterestKey"
            ),
            suspension_4_df.ACCOUNT_NUMBER.cast("string").alias("ExposureKey"),
            suspension_4_df.CUST_ID.cast("string").alias("PartyKey"),
            suspension_4_df.CDI.alias("CDI"),
            suspension_4_df.InterestAmount.cast("string").alias("Interest"),
            suspension_4_df.PenaltyAmount.cast("string").alias("PenaltyInterest"),
            suspension_4_df.StartDate.alias("StartDate"),
            suspension_4_df.businessdate.alias("BusinessDate"),
        )

        # ==========================  suspension_5_df =====================================

        # Rename multiple column fields to avoid issues that causes ambiguity

        user_defined_fields_suspension_5 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_prime")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_prime")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_prime")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_prime")
        )

        user_defined_fields_t1_suspension_5 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_t1")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_t1")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_t1")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_t1")
        )

        user_defined_fields_t2_suspension_5 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_t2")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_t2")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_t2")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_t2")
        )

        user_defined_fields_dt_suspension_5 = (
            user_defined_fields.withColumnRenamed("PFG_TAG", "PFG_TAG_dt")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_dt")
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_dt")
            .withColumnRenamed("FIELD_VALUE", "FIELD_VALUE_dt")
        )

        # Join user_defined_fields with itself

        user_defined_fields_suspension_5 = user_defined_fields_suspension_5.filter(
            (user_defined_fields_suspension_5.PFG_TAG_prime == "PFTAG109")
            & (
                user_defined_fields_suspension_5.PFG_TAG_SET_CODE_prime
                == "ΤΟΚΟΙ Ν.3869"
            )
        )

        user_defined_fields_dt_suspension_5_join_condition = (
            (
                user_defined_fields_suspension_5.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_dt_suspension_5.UD_ACCOUNT_NUMBER_dt
            )
            & (
                user_defined_fields_dt_suspension_5.PFG_TAG_SET_CODE_dt
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_dt_suspension_5.PFG_TAG_dt == "PFTAG110")
        )

        user_defined_fields_t1_suspension_5_join_condition = (
            (
                user_defined_fields_suspension_5.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_t1_suspension_5.UD_ACCOUNT_NUMBER_t1
            )
            & (
                user_defined_fields_t1_suspension_5.PFG_TAG_SET_CODE_t1
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_t1_suspension_5.PFG_TAG_t1 == "PFTAG111")
        )

        user_defined_fields_t2_suspension_5_join_condition = (
            (
                user_defined_fields_suspension_5.UD_ACCOUNT_NUMBER_prime
                == user_defined_fields_t2_suspension_5.UD_ACCOUNT_NUMBER_t2
            )
            & (
                user_defined_fields_t2_suspension_5.PFG_TAG_SET_CODE_t2
                == "ΤΟΚΟΙ Ν.3869"
            )
            & (user_defined_fields_t2_suspension_5.PFG_TAG_t2 == "PFTAG112")
        )

        portfolio_loan_details_join_condition = (
            portfolio.unit_code_profits == loan_details.prft_acc_unit
        )

        suspension_5_df = (
            user_defined_fields_suspension_5.join(
                user_defined_fields_dt_suspension_5,
                user_defined_fields_dt_suspension_5_join_condition,
                "inner",
            )
            .join(
                user_defined_fields_t1_suspension_5,
                user_defined_fields_t1_suspension_5_join_condition,
                "inner",
            )
            .join(
                user_defined_fields_t2_suspension_5,
                user_defined_fields_t2_suspension_5_join_condition,
                "inner",
            )
        )

        suspension_5_df = suspension_5_df.select(
            user_defined_fields_suspension_5.UD_ACCOUNT_NUMBER_prime.alias(
                "ACCOUNT_NUMBER"
            ),
            user_defined_fields_suspension_5.FIELD_VALUE_prime.alias("cust_id_5"),
            user_defined_fields_dt_suspension_5.FIELD_VALUE_dt.alias("StartDate"),
            user_defined_fields_t1_suspension_5.FIELD_VALUE_t1.alias("InterestAmount"),
            user_defined_fields_t2_suspension_5.FIELD_VALUE_t2.alias("PenaltyAmount"),
            user_defined_fields_suspension_5.PFG_TAG_prime.alias("PFG_TAG"),
            user_defined_fields_suspension_5.BusinessDate.alias("businessdate"),
        )

        suspension_5_df = suspension_5_df.join(
            customer, suspension_5_df.cust_id_5 == customer.CUST_ID, "inner"
        )

        # creating another table from user_defined_fields with some specific filter conditions in order to satisfy the where exists criterion

        user_defined_fields_a_5 = user_defined_fields.filter(
            (user_defined_fields.PFG_TAG == "PFTAG141")
            & (user_defined_fields.FIELD_VALUE.isin(["04", "05", "20"]))
        )

        user_defined_fields_a_5 = user_defined_fields_a_5.withColumn(
            "UD_ACCOUNT_NUMBER_INT",
            user_defined_fields_a_5.UD_ACCOUNT_NUMBER.cast("int"),
        )

        # join conditions
        suspension_5_df_loan_details_join_condition = (
            suspension_5_df.ACCOUNT_NUMBER == loan_details.account_number_ld
        )
        suspension_df_5_df_user_defined_fields_a_5_join_condition = (
            user_defined_fields_a_5.UD_ACCOUNT_NUMBER_INT == suspension_5_df.cust_id_5
        )

        # loan_details_agreement_join_condition=(loan_details.agreement_number == agreement.agreement_number)

        suspension_5_df = (
            suspension_5_df.join(
                user_defined_fields_a_5,
                suspension_df_5_df_user_defined_fields_a_5_join_condition,
                "left_semi",
            )
            .join(loan_details, suspension_5_df_loan_details_join_condition, "inner")
            .join(
                portfolio,
                portfolio_loan_details_join_condition,
                "left",
            )
        )
        # .join(
        #    agreement,
        #    loan_details_agreement_join_condition,
        #    "left"
        # )

        L3869SuspensionInterestId_5_primary_key_fields = [
            suspension_5_df.ACCOUNT_NUMBER,
            suspension_5_df.CUST_ID,
            lit("profits"),
        ]

        L3869SuspensionInterestKey_5_primary_key_fields = [
            suspension_5_df.ACCOUNT_NUMBER,
            suspension_5_df.CUST_ID,
        ]

        suspension_5_df = suspension_5_df.withColumn(
            "L3869SuspensionInterestId",
            sha2(concat_ws(";", *L3869SuspensionInterestId_5_primary_key_fields), 256),
        ).withColumn(
            "L3869SuspensionInterestKey",
            concat_ws(";", *L3869SuspensionInterestKey_5_primary_key_fields),
        )

        suspension_5_target_df = suspension_5_df.select(
            suspension_5_df.L3869SuspensionInterestId.alias(
                "L3869SuspensionInterestId"
            ),
            lit("profits").alias("SourceSystem"),
            portfolio.unit_code_profits.alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            suspension_5_df.L3869SuspensionInterestKey.alias(
                "L3869SuspensionInterestKey"
            ),
            suspension_5_df.ACCOUNT_NUMBER.cast("string").alias("ExposureKey"),
            suspension_5_df.CUST_ID.cast("string").alias("PartyKey"),
            suspension_5_df.CDI.alias("CDI"),
            suspension_5_df.InterestAmount.cast("string").alias("Interest"),
            suspension_5_df.PenaltyAmount.cast("string").alias("PenaltyInterest"),
            suspension_5_df.StartDate.alias("StartDate"),
            suspension_5_df.businessdate.alias("BusinessDate"),
        )

        # ========================= Append ====================

        l3869suspensioninterest = (
            suspension_1_target_df.union(suspension_2_target_df)
            .union(suspension_3_target_df)
            .union(suspension_4_target_df)
            .union(suspension_5_target_df)
        )

        l3869suspensioninterest = l3869suspensioninterest.distinct()

        l3869suspensioninterest.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_l3869_suspension_interest_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)